# YOLOv11 Volleyball Detection

**Pipeline:** Dataset Download → Training → Validation → Testing → CSV Metrics Export

**Classes:**
- 0: ball
- 1: player
- 2: player 1
- 3: referee

## 1. Setup & Install Dependencies

In [1]:
!pip install roboflow ultralytics torch torchvision

In [2]:
import os
import sys
import csv
import logging
from datetime import datetime

import torch
from ultralytics import YOLO
from roboflow import Roboflow

print(f"Python      : {sys.version}")
print(f"PyTorch     : {torch.__version__}")
print(f"CUDA avail  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU         : {torch.cuda.get_device_name(0)}")
    print(f"VRAM        : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")

Python      : 3.12.11 | packaged by Anaconda, Inc. | (main, Jun  5 2025, 13:09:17) [GCC 11.2.0]
PyTorch     : 2.5.1+cu121
CUDA avail  : True
GPU         : Tesla T4
VRAM        : 14.6 GB


## 2. Download Dataset from Roboflow

In [3]:
rf = Roboflow(api_key="MUEDCloFvcri2QqbuoB0")
project = rf.workspace("jeevaraj-s").project("volleyball-hwxp2-wq92v")
version = project.version(2)
dataset = version.download("yolov11")

print(f"\n✅ Dataset downloaded to: {dataset.location}")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to volleyball-2 in yolov11:: 100%|██████████| 314/314 [00:00<00:00, 4059.80it/s]


✅ Dataset downloaded to: /teamspace/studios/this_studio/Volleyball01/YOLOv11/volleyball-2


## 3. Configure Paths

In [4]:
# ────────────────────────── CONFIGURATION ──────────────────────────
# Notebook directory = YOLOv11 folder
SCRIPT_DIR = os.path.abspath(os.getcwd())
BASE_DIR   = os.path.dirname(SCRIPT_DIR)

# Dataset location (from Roboflow download)
DATASET_DIR = os.path.abspath(dataset.location)
DATA_YAML   = os.path.join(DATASET_DIR, "data.yaml")

# Model — will be auto-downloaded by ultralytics if not present
MODEL_PATH = os.path.join(BASE_DIR, "yolo11n.pt")

# Training hyper-params
EPOCHS      = 100
IMG_SIZE    = 640
BATCH_SIZE  = 4       # safe for RTX 3050 4 GB VRAM
PATIENCE    = 50      # early-stopping patience
SAVE_PERIOD = 10      # checkpoint every N epochs

# Output directories
PROJECT_DIR = os.path.join(SCRIPT_DIR, "runs")
LOG_DIR     = os.path.join(SCRIPT_DIR, "logs")

print(f"SCRIPT_DIR  : {SCRIPT_DIR}")
print(f"BASE_DIR    : {BASE_DIR}")
print(f"DATASET_DIR : {DATASET_DIR}")
print(f"DATA_YAML   : {DATA_YAML}")
print(f"MODEL_PATH  : {MODEL_PATH}")
print(f"PROJECT_DIR : {PROJECT_DIR}")

SCRIPT_DIR  : /teamspace/studios/this_studio/Volleyball01/YOLOv11
BASE_DIR    : /teamspace/studios/this_studio/Volleyball01
DATASET_DIR : /teamspace/studios/this_studio/Volleyball01/YOLOv11/volleyball-2
DATA_YAML   : /teamspace/studios/this_studio/Volleyball01/YOLOv11/volleyball-2/data.yaml
MODEL_PATH  : /teamspace/studios/this_studio/Volleyball01/yolo11n.pt
PROJECT_DIR : /teamspace/studios/this_studio/Volleyball01/YOLOv11/runs


In [5]:
# Update data.yaml with absolute path to the dataset directory
def update_data_yaml(data_yaml_path, dataset_dir):
    """Update data.yaml with absolute path to dataset directory."""
    with open(data_yaml_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    updated_lines = []
    for line in lines:
        if line.strip().startswith('path:'):
            updated_lines.append(f'path: {dataset_dir}\n')
        else:
            updated_lines.append(line)

    with open(data_yaml_path, 'w', encoding='utf-8') as f:
        f.writelines(updated_lines)

    print(f"✅ Updated data.yaml with path: {dataset_dir}")

if os.path.isfile(DATA_YAML):
    update_data_yaml(DATA_YAML, DATASET_DIR)
    # Print data.yaml contents for verification
    with open(DATA_YAML, 'r') as f:
        print("\n--- data.yaml ---")
        print(f.read())
else:
    print(f"❌ data.yaml not found at: {DATA_YAML}")

✅ Updated data.yaml with path: /teamspace/studios/this_studio/Volleyball01/YOLOv11/volleyball-2

--- data.yaml ---
train: ../train/images
val: ../valid/images
test: ../test/images

nc: 4
names: ['ball', 'player', 'player 1', 'referee']

roboflow:
  workspace: jeevaraj-s
  project: volleyball-hwxp2-wq92v
  version: 2
  license: CC BY 4.0
  url: https://universe.roboflow.com/jeevaraj-s/volleyball-hwxp2-wq92v/dataset/2


## 4. GPU Check

In [6]:
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

if torch.cuda.is_available():
    DEVICE = 0
    name = torch.cuda.get_device_name(0)
    mem  = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"✅ GPU detected: {name}  ({mem:.1f} GB VRAM)")
else:
    DEVICE = "cpu"
    print("❌ No GPU found — falling back to CPU (training will be slow).")

print(f"Using device: {DEVICE}")

✅ GPU detected: Tesla T4  (14.6 GB VRAM)
Using device: 0


## 5. Setup Logger

In [7]:
os.makedirs(LOG_DIR, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
LOG_FILE  = os.path.join(LOG_DIR, f"training_{timestamp}.log")

logger = logging.getLogger("yolo_train")
logger.setLevel(logging.INFO)
logger.handlers.clear()

# Console handler
ch = logging.StreamHandler(sys.stdout)
ch.setLevel(logging.INFO)
ch.setFormatter(logging.Formatter("%(message)s"))
logger.addHandler(ch)

# File handler
fh = logging.FileHandler(LOG_FILE, encoding="utf-8")
fh.setLevel(logging.INFO)
fh.setFormatter(logging.Formatter("%(asctime)s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S"))
logger.addHandler(fh)

logger.info(f"Log file → {LOG_FILE}")

Log file → /teamspace/studios/this_studio/Volleyball01/YOLOv11/logs/training_20260329_090843.log


## 6. Train YOLOv11

In [8]:
RUN_NAME = "volleyball_train"

logger.info("=" * 60)
logger.info("  YOLOv11 Volleyball Detection — GPU Training")
logger.info("=" * 60)
logger.info(f"  Model      : {MODEL_PATH}")
logger.info(f"  Dataset    : {DATA_YAML}")
logger.info(f"  Epochs     : {EPOCHS}")
logger.info(f"  Image size : {IMG_SIZE}")
logger.info(f"  Batch size : {BATCH_SIZE}")
logger.info(f"  Patience   : {PATIENCE}")
logger.info(f"  Output     : {PROJECT_DIR}/{RUN_NAME}")
logger.info("=" * 60)

# Load model
logger.info("Loading model …")
model = YOLO(MODEL_PATH)

# Train
logger.info("Starting training …\n")
train_results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    device=DEVICE,
    batch=BATCH_SIZE,
    workers=0,
    patience=PATIENCE,

    # Auto-save settings
    save=True,
    save_period=SAVE_PERIOD,
    project=PROJECT_DIR,
    name=RUN_NAME,
    exist_ok=True,

    # Generate all plots / charts automatically
    plots=True,

    # Augmentation (tuned for volleyball + players)
    mosaic=1.0,
    mixup=0.1,
    scale=0.5,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,

    cache=False,
    verbose=True,
)

  YOLOv11 Volleyball Detection — GPU Training
  Model      : /teamspace/studios/this_studio/Volleyball01/yolo11n.pt
  Dataset    : /teamspace/studios/this_studio/Volleyball01/YOLOv11/volleyball-2/data.yaml
  Epochs     : 100
  Image size : 640
  Batch size : 4
  Patience   : 50
  Output     : /teamspace/studios/this_studio/Volleyball01/YOLOv11/runs/volleyball_train
Loading model …
Starting training …

New https://pypi.org/project/ultralytics/8.4.31 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.16 🚀 Python-3.12.11 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/teamspace/studios/this_studio/Volleyball01/YOLOv11/volleyball-2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, d

In [9]:
# ── Print Training Summary ──
output_dir = os.path.join(PROJECT_DIR, RUN_NAME)

logger.info("\n" + "=" * 60)
logger.info("  TRAINING COMPLETE — FINAL METRICS")
logger.info("=" * 60)

train_metrics = {}
if train_results and hasattr(train_results, "results_dict"):
    metrics = train_results.results_dict
    logger.info(f"\n{'Metric':<35} {'Value':>10}")
    logger.info("-" * 47)
    for key, value in metrics.items():
        if isinstance(value, float):
            logger.info(f"{key:<35} {value:>10.4f}")
        else:
            logger.info(f"{key:<35} {str(value):>10}")
    train_metrics = metrics

logger.info(f"\n📁 Results saved to : {output_dir}")
logger.info("📊 Auto-saved charts:")
for chart in [
    "results.png", "confusion_matrix.png", "confusion_matrix_normalized.png",
    "F1_curve.png", "P_curve.png", "R_curve.png", "PR_curve.png",
    "labels.jpg", "labels_correlogram.jpg",
]:
    path = os.path.join(output_dir, chart)
    status = "✅" if os.path.isfile(path) else "—"
    logger.info(f"   {status} {chart}")

logger.info(f"\n🏆 Best weights : {output_dir}/weights/best.pt")
logger.info(f"   Last weights : {output_dir}/weights/last.pt")
logger.info("=" * 60)


  TRAINING COMPLETE — FINAL METRICS

Metric                                   Value
-----------------------------------------------
metrics/precision(B)                    0.9893
metrics/recall(B)                       0.8711
metrics/mAP50(B)                        0.9123
metrics/mAP50-95(B)                     0.7285
fitness                                 0.7285

📁 Results saved to : /teamspace/studios/this_studio/Volleyball01/YOLOv11/runs/volleyball_train
📊 Auto-saved charts:
   ✅ results.png
   ✅ confusion_matrix.png
   ✅ confusion_matrix_normalized.png
   — F1_curve.png
   — P_curve.png
   — R_curve.png
   — PR_curve.png
   ✅ labels.jpg
   — labels_correlogram.jpg

🏆 Best weights : /teamspace/studios/this_studio/Volleyball01/YOLOv11/runs/volleyball_train/weights/best.pt
   Last weights : /teamspace/studios/this_studio/Volleyball01/YOLOv11/runs/volleyball_train/weights/last.pt


## 7. Validate YOLOv11

In [10]:
# Load best weights for validation
BEST_MODEL_PATH = os.path.join(PROJECT_DIR, RUN_NAME, "weights", "best.pt")

if not os.path.exists(BEST_MODEL_PATH):
    print(f"❌ Error: Model weights not found at {BEST_MODEL_PATH}")
    print("   Please ensure training completed successfully.")
else:
    print(f"✅ Loading best model from: {BEST_MODEL_PATH}")
    val_model = YOLO(BEST_MODEL_PATH)

    val_project_dir = os.path.join(SCRIPT_DIR, "runs", "val")
    val_name = "volleyball_val"

    print(f"🚀 Starting Validation on dataset: {DATA_YAML}")

    val_metrics = val_model.val(
        data=DATA_YAML,
        project=val_project_dir,
        name=val_name,
        save_json=True,
        save_txt=True,
        save_conf=True,
        plots=True,
        verbose=True,
    )

    # Extract metrics
    try:
        val_map50 = float(val_metrics.box.map50)
        val_map95 = float(val_metrics.box.map)
    except Exception:
        val_map50, val_map95 = 0.0, 0.0

    try:
        if hasattr(val_metrics.box, 'mp') and hasattr(val_metrics.box, 'mr'):
            val_p = float(val_metrics.box.mp)
            val_r = float(val_metrics.box.mr)
        elif hasattr(val_metrics.box, 'mean_results'):
            res = val_metrics.box.mean_results()
            val_p, val_r = float(res[0]), float(res[1])
        else:
            val_p = float(sum(val_metrics.box.p) / len(val_metrics.box.p)) if len(val_metrics.box.p) > 0 else 0.0
            val_r = float(sum(val_metrics.box.r) / len(val_metrics.box.r)) if len(val_metrics.box.r) > 0 else 0.0
    except Exception:
        val_p, val_r = 0.0, 0.0

    print("\n" + "━" * 50)
    print("📈 YOLOv11 VALIDATION METRICS")
    print("━" * 50)
    print(f"Precision:   {val_p:.4f}")
    print(f"Recall:      {val_r:.4f}")
    print(f"mAP@50:      {val_map50:.4f}")
    print(f"mAP@50-95:   {val_map95:.4f}")
    print("━" * 50)
    print(f"📂 Results saved in: {os.path.join(val_project_dir, val_name)}")

✅ Loading best model from: /teamspace/studios/this_studio/Volleyball01/YOLOv11/runs/volleyball_train/weights/best.pt
🚀 Starting Validation on dataset: /teamspace/studios/this_studio/Volleyball01/YOLOv11/volleyball-2/data.yaml
Ultralytics 8.4.16 🚀 Python-3.12.11 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n summary (fused): 101 layers, 2,582,932 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2540.4±604.4 MB/s, size: 92.0 KB)
val: Scanning /teamspace/studios/this_studio/Volleyball01/YOLOv11/volleyball-2/valid/labels.cache... 27 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 27/27 9.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.7s/it 5.5s<10.1s
                   all         27        293      0.989      0.872      0.913      0.726
                  ball         15         15          1      0.661      0.755      0.481
                player         27       

## 8. Test YOLOv11

In [11]:
# Load best weights for testing
if not os.path.exists(BEST_MODEL_PATH):
    print(f"❌ Error: Model weights not found at {BEST_MODEL_PATH}")
else:
    print(f"✅ Loading model from: {BEST_MODEL_PATH}")
    test_model = YOLO(BEST_MODEL_PATH)

    test_project_dir = os.path.join(SCRIPT_DIR, "runs", "val")
    test_name = "volleyball_test"

    print(f"🚀 Starting Testing on dataset: {DATA_YAML}")

    test_metrics = test_model.val(
        data=DATA_YAML,
        split="test",
        project=test_project_dir,
        name=test_name,
        save_json=True,
        save_txt=True,
        save_conf=True,
        plots=True,
        verbose=True,
    )

    # Extract metrics
    try:
        test_map50 = float(test_metrics.box.map50)
        test_map95 = float(test_metrics.box.map)
    except Exception:
        test_map50, test_map95 = 0.0, 0.0

    try:
        if hasattr(test_metrics.box, 'mp') and hasattr(test_metrics.box, 'mr'):
            test_p = float(test_metrics.box.mp)
            test_r = float(test_metrics.box.mr)
        elif hasattr(test_metrics.box, 'mean_results'):
            res = test_metrics.box.mean_results()
            test_p, test_r = float(res[0]), float(res[1])
        else:
            test_p = float(sum(test_metrics.box.p) / len(test_metrics.box.p)) if len(test_metrics.box.p) > 0 else 0.0
            test_r = float(sum(test_metrics.box.r) / len(test_metrics.box.r)) if len(test_metrics.box.r) > 0 else 0.0
    except Exception:
        test_p, test_r = 0.0, 0.0

    print("\n" + "━" * 50)
    print("📈 YOLOv11 TEST METRICS")
    print("━" * 50)
    print(f"Precision:   {test_p:.4f}")
    print(f"Recall:      {test_r:.4f}")
    print(f"mAP@50:      {test_map50:.4f}")
    print(f"mAP@50-95:   {test_map95:.4f}")
    print("━" * 50)
    print(f"📂 Results saved in: {os.path.join(test_project_dir, test_name)}")

✅ Loading model from: /teamspace/studios/this_studio/Volleyball01/YOLOv11/runs/volleyball_train/weights/best.pt
🚀 Starting Testing on dataset: /teamspace/studios/this_studio/Volleyball01/YOLOv11/volleyball-2/data.yaml
Ultralytics 8.4.16 🚀 Python-3.12.11 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n summary (fused): 101 layers, 2,582,932 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 105.5±11.7 MB/s, size: 101.6 KB)
val: Scanning /teamspace/studios/this_studio/Volleyball01/YOLOv11/volleyball-2/test/labels... 6 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 6/6 1.2Kit/s 0.0s
val: New cache created: /teamspace/studios/this_studio/Volleyball01/YOLOv11/volleyball-2/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.4s/it 1.4s
                   all          6         90      0.983      0.933      0.945      0.699
                  ball          6          6   

## 9. Save All Metrics to CSV

In [12]:
CSV_PATH = os.path.join(SCRIPT_DIR, "metrics_summary.csv")

# Collect training final metrics (from results_dict)
try:
    train_p    = train_metrics.get("metrics/precision(B)", 0.0)
    train_r    = train_metrics.get("metrics/recall(B)", 0.0)
    train_m50  = train_metrics.get("metrics/mAP50(B)", 0.0)
    train_m95  = train_metrics.get("metrics/mAP50-95(B)", 0.0)
except Exception:
    train_p = train_r = train_m50 = train_m95 = 0.0

rows = [
    ["Phase", "Precision", "Recall", "mAP50", "mAP50-95"],
    ["Train",  f"{train_p:.4f}",  f"{train_r:.4f}",  f"{train_m50:.4f}",  f"{train_m95:.4f}"],
    ["Valid",  f"{val_p:.4f}",    f"{val_r:.4f}",    f"{val_map50:.4f}",  f"{val_map95:.4f}"],
    ["Test",   f"{test_p:.4f}",   f"{test_r:.4f}",   f"{test_map50:.4f}", f"{test_map95:.4f}"],
]

with open(CSV_PATH, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerows(rows)

print(f"✅ Metrics saved to: {CSV_PATH}")
print()

# Pretty-print the table
print(f"{'Phase':<8} {'Precision':>10} {'Recall':>10} {'mAP50':>10} {'mAP50-95':>10}")
print("─" * 52)
for row in rows[1:]:
    print(f"{row[0]:<8} {row[1]:>10} {row[2]:>10} {row[3]:>10} {row[4]:>10}")

✅ Metrics saved to: /teamspace/studios/this_studio/Volleyball01/YOLOv11/metrics_summary.csv

Phase     Precision     Recall      mAP50   mAP50-95
────────────────────────────────────────────────────
Train        0.9893     0.8711     0.9123     0.7285
Valid        0.9889     0.8716     0.9126     0.7262
Test         0.9829     0.9333     0.9450     0.6987


In [13]:
# ── Final Summary ──
print("\n" + "=" * 60)
print("  YOLOv11 VOLLEYBALL PIPELINE COMPLETE")
print("=" * 60)
print(f"  📂 Training results : {os.path.join(PROJECT_DIR, RUN_NAME)}")
print(f"  📂 Validation results : {os.path.join(SCRIPT_DIR, 'runs', 'val', 'volleyball_val')}")
print(f"  📂 Test results     : {os.path.join(SCRIPT_DIR, 'runs', 'val', 'volleyball_test')}")
print(f"  📊 Metrics CSV      : {CSV_PATH}")
print(f"  📝 Training log     : {LOG_FILE}")
print(f"  🏆 Best weights     : {BEST_MODEL_PATH}")
print("=" * 60)


  YOLOv11 VOLLEYBALL PIPELINE COMPLETE
  📂 Training results : /teamspace/studios/this_studio/Volleyball01/YOLOv11/runs/volleyball_train
  📂 Validation results : /teamspace/studios/this_studio/Volleyball01/YOLOv11/runs/val/volleyball_val
  📂 Test results     : /teamspace/studios/this_studio/Volleyball01/YOLOv11/runs/val/volleyball_test
  📊 Metrics CSV      : /teamspace/studios/this_studio/Volleyball01/YOLOv11/metrics_summary.csv
  📝 Training log     : /teamspace/studios/this_studio/Volleyball01/YOLOv11/logs/training_20260329_090843.log
  🏆 Best weights     : /teamspace/studios/this_studio/Volleyball01/YOLOv11/runs/volleyball_train/weights/best.pt
